# Create BMWE Awards (German Federal Ministry for Economic Affairs and Energy)

Creates awards for the **Bundesministerium für Wirtschaft und Energie (BMWE)** -- BMWi until 2021,
BMWK (Wirtschaft und Klimaschutz) 2021-2025, BMWE again since 2025 -- from the federal **Förderkatalog**
(https://foerderportal.bund.de/foekat). 48,391 projects on 2026-10-01 (completed included), mostly
energy research (PT-J), aviation/maritime (PT-DLR, PT-TÜV), industrial collective research IGF and
EXIST university spin-offs.

**Twins:** F4303701741 (BMWK) and F4320321469 (BMWi, Wirtschaft und Technologie) share ROR 02vgg2808
and Wikidata Q488589 with F4320323803; their citation stubs do not collapse here (funder-merge
follow-up, #1491).

**Prerequisites:** `scripts/local/foekat_to_s3.py --ressort BMWE` (detail search Ressort = BMWE
including completed projects, the catalogue's own CSV export "Ausgabe als Textdatei"; uploads
`s3://openalex-ingest/awards/foekat/bmwe_projects.parquet` with the §1.4 shrink guard).

**Funder (Path A, F4320*):** funder_id `4320323803` (Bundesministerium für Wirtschaft und Energie), read from `openalex.funders.funders`.
The catalogue lists completed projects under the ministry responsible **today**, so projects of the
ministry's earlier incarnations are included under this row.

**Scope filter:** none beyond the scraper's bookkeeping drop.

**Legacy BMBF rows (oxjob #1491).** Until 2026-10 the Förderkatalog was ingested only through
`CreateBMBFAwards`, which put every row under BMBF. Those 45,998 rows (a 2026-01-27 scrape, frozen at
`s3://openalex-ingest/awards/foekat/bmbf_legacy_published_20260127.parquet`) are published stable award ids and must
not vanish from `openalex_awards_raw` (stable-id completeness check). 9,377 of them belong to other
ministries (BMWE 6,751, BMV 2,575, BMUKN 51); moving them needs Rohan's stable-id path (#1491 phase 2).
Until then they stay under BMBF, and the ministry notebooks **exclude** their FKZs so no project
appears under two funders. The old scraper read FKZs from result-link URLs, so a space became `+`
(`BCT0006+/2`, catalogue `BCT0006 /2`); matching is on `LOWER(REPLACE(fkz, ' ', '+'))`.

**Fields:** `funder_award_id` = FKZ (Förderkennzeichen, the reference papers cite). `amount` = federal share
in **EUR** (EUR 0 -> NULL). No PI names are published: `lead_investigator` carries only the grantee
institution, NULL when the catalogue withholds it ("Keine Anzeige ..."). `funder_scheme` = the
Leistungsplansystematik text. `funding_type` = grant (EXIST-Gründerstipendium = fellowship).

**Priority:** `550` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.foekat_bmwe_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/foekat/bmwe_projects.parquet`;

In [ ]:
%sql
SELECT ressort, COUNT(*) as total_projects, COUNT(DISTINCT LOWER(fkz)) as distinct_fkz
FROM openalex.awards.foekat_bmwe_raw GROUP BY ressort;

In [ ]:
%sql
SELECT admin_agency, lps_text, COUNT(*) AS n, ROUND(SUM(TRY_CAST(amount_eur AS DOUBLE)), 0) AS eur
FROM openalex.awards.foekat_bmwe_raw
GROUP BY 1, 2 ORDER BY n DESC LIMIT 50;

## Step 1.6: Funder existence check (Path A)
F4320323803 must resolve to exactly one row in `openalex.funders.funders`; if not, STOP.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'funder 4320323803 must resolve to exactly one row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320323803;

## Step 2: Create BMWE Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bmwe_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323803  -- Bundesministerium für Wirtschaft und Energie
),
-- FKZs still published under BMBF (CreateBMBFAwards legacy rows, #1491): left out here until phase 2.
legacy AS (
    SELECT DISTINCT LOWER(fkz) AS legacy_key
    FROM parquet.`s3a://openalex-ingest/awards/foekat/bmbf_legacy_published_20260127.parquet`
    WHERE fkz IS NOT NULL AND TRIM(CAST(fkz AS STRING)) != ''
),
g AS (
    SELECT
        TRIM(c.fkz) AS funder_award_id,
        NULLIF(TRIM(c.title), '') AS title,
        TRY_CAST(c.amount_eur AS DOUBLE) AS amount,
        NULLIF(TRIM(c.lps_text), '') AS scheme,
        CASE WHEN c.title LIKE 'EXIST-Gründerstipendium%' THEN 'fellowship' ELSE 'grant' END AS funding_type,
        TRY_TO_DATE(c.start_date, 'yyyy-MM-dd') AS start_d,
        TRY_TO_DATE(c.end_date, 'yyyy-MM-dd') AS end_d,
        CASE WHEN NULLIF(TRIM(c.recipient), '') IS NOT NULL AND c.recipient NOT LIKE 'Keine Anzeige%' THEN TRIM(c.recipient) END AS inst,
        NULLIF(TRIM(c.recipient_country_iso), '') AS inst_country,
        c.landing_page_url
    FROM openalex.awards.foekat_bmwe_raw c
    LEFT ANTI JOIN legacy l ON l.legacy_key = LOWER(REPLACE(TRIM(c.fkz), ' ', '+'))
    WHERE c.fkz IS NOT NULL AND TRIM(c.fkz) != '' AND c.ressort = 'BMWE'
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    g.title as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type,
    g.scheme as funder_scheme,
    'foerderkatalog_bmwe' as provenance,
    g.start_d as start_date,
    g.end_d as end_date,
    -- Future-dated placeholders (start more than a year ahead) get no years; see the
    -- how-to-add-a-funder "Future-dated start/end years" note.
    CASE WHEN YEAR(g.start_d) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(g.start_d) END as start_year,
    CASE WHEN YEAR(g.start_d) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(g.end_d) END as end_year,
    -- No PI names in the Förderkatalog: institution-only lead (BMEL precedent);
    -- struct field order MUST match openalex_awards_raw.
    CASE WHEN g.inst IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.inst,
            'country', g.inst_country,
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Guards: one row per FKZ, non-empty, and no FKZ that is already under BMBF (provenance
-- 'foerderkatalog', priority 36) -- a project must never sit under two ministries (#1491).
SELECT
    -- One row per FKZ. Raw ids (xxhash64 % 9e9) can collide; the stable-id allocator recovers those
    -- (COLLISION_RECOVERY), so duplicate ids are reported, not blocked.
    assert_true((SELECT COUNT(*) FROM openalex.awards.bmwe_awards) =
                (SELECT COUNT(DISTINCT LOWER(funder_award_id)) FROM openalex.awards.bmwe_awards), 'duplicate FKZ') AS fkz_unique,
    (SELECT COUNT(*) - COUNT(DISTINCT id) FROM openalex.awards.bmwe_awards) AS raw_id_collisions,
    assert_true((SELECT COUNT(*) FROM openalex.awards.bmwe_awards) > 0, 'empty build') AS non_empty,
    assert_true((
        SELECT COUNT(*) FROM openalex.awards.bmwe_awards a
        JOIN openalex.awards.openalex_awards_raw r
          ON LOWER(REPLACE(r.funder_award_id, ' ', '+')) = LOWER(REPLACE(a.funder_award_id, ' ', '+'))
        WHERE r.provenance = 'foerderkatalog' AND r.priority = 36
    ) = 0, 'FKZ already under BMBF (foerderkatalog, 36)') AS not_under_bmbf,
    (SELECT COUNT(*) FROM openalex.awards.bmwe_awards) AS n;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'foerderkatalog_bmwe' AND priority = 550;

-- Insert into openalex_awards_raw with priority 550 (direct funder ingest; higher wins
-- under the 2026-06-20 DESC dedup, so it outranks the priority-0 acknowledgement shells).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    550 as priority
FROM openalex.awards.bmwe_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids, ROUND(SUM(amount), 0) AS total_eur
FROM openalex.awards.bmwe_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_date, end_date,
       lead_investigator.affiliation.name AS institution, landing_page_url
FROM openalex.awards.bmwe_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.bmwe_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 30;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.bmwe_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency checks (no PI names published -> grantee institution and title).
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) AS n
FROM openalex.awards.bmwe_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.bmwe_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_institution
FROM openalex.awards.bmwe_awards;

In [ ]:
%sql
-- Citation stubs (priority < 3) that collapse onto this ingest, by funder row.
SELECT c.funder_id, COUNT(*) as stub_rows, COUNT(t.id) as collapse_onto_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.bmwe_awards t ON t.id = c.id
WHERE c.funder_id = 4320323803 AND c.priority < 3
GROUP BY 1;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'foerderkatalog_bmwe'
GROUP BY provenance, priority;